# BeyondBench: Result Analysis

This notebook provides a deep-dive into analyzing BeyondBench evaluation results.

**What you'll learn:**
- How to load and parse result JSON files
- Aggregate performance across tasks and suites
- Visualize distributions, trends, and per-task breakdowns
- Statistical analysis of model performance

**Prerequisites:**
```bash
pip install beyondbench matplotlib pandas numpy scipy
```

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

print("Dependencies loaded successfully")

## 1. Generate Synthetic Evaluation Results

We simulate a realistic set of results for analysis. Replace `synthetic_results` with your actual `final_results.json` data.

In [ ]:
np.random.seed(42)

# Task definitions matching actual BeyondBench suites
EASY_TASKS = [
    "sum", "mean", "sorting", "find_maximum", "find_minimum", "comparison",
    "odd_count", "even_count", "median", "mode", "absolute_difference",
    "division", "multiplication", "subtraction", "second_maximum", "range",
    "count_negative", "count_unique", "reverse_list", "dot_product",
]

MEDIUM_TASKS = [
    "fibonacci_sequence", "geometric_sequence", "arithmetic_progression",
    "polynomial_evaluation", "matrix_operations", "combinatorics",
    "number_base_conversion", "logical_operations", "gcd_lcm", "collatz_sequence",
]

HARD_TASKS = [
    "knapsack", "shortest_path", "edit_distance", "coin_change",
    "longest_common_subsequence", "traveling_salesman", "topological_sort",
    "interval_scheduling", "n_queens", "sudoku_solving",
    "boolean_sat", "graph_coloring", "tower_hanoi", "cryptarithmetic",
]

SUITE_TASKS = {"easy": EASY_TASKS, "medium": MEDIUM_TASKS, "hard": HARD_TASKS}

def simulate_task_results(base_accuracy: float, noise: float = 0.05, n_datapoints: int = 100):
    """Simulate per-datapoint results for a task."""
    acc = np.clip(base_accuracy + np.random.normal(0, noise), 0, 1)
    n_correct = int(round(acc * n_datapoints))
    samples = [True] * n_correct + [False] * (n_datapoints - n_correct)
    np.random.shuffle(samples)
    return {
        "accuracy": acc,
        "total_samples": n_datapoints,
        "correct_samples": n_correct,
        "samples": samples,
    }

# Simulate a GPT-4o evaluation
task_results = {}
base_accuracies = {
    **{t: np.random.uniform(0.88, 0.99) for t in EASY_TASKS},
    **{t: np.random.uniform(0.72, 0.93) for t in MEDIUM_TASKS},
    **{t: np.random.uniform(0.45, 0.82) for t in HARD_TASKS},
}

for task in EASY_TASKS + MEDIUM_TASKS + HARD_TASKS:
    task_results[task] = [simulate_task_results(base_accuracies[task])]

synthetic_results = {
    "model_info": {
        "model_name": "gpt-4o",
        "backend": "openai",
        "api_provider": "openai",
        "model_type": "api_based",
    },
    "summary": {
        "total_tasks": len(task_results),
        "completed_tasks": len(task_results),
        "avg_accuracy": np.mean([r[0]["accuracy"] for r in task_results.values()]),
        "total_duration": 1842.5,
    },
    "task_results": task_results,
}

print(f"Simulated results for {len(task_results)} tasks")
print(f"Overall average accuracy: {synthetic_results['summary']['avg_accuracy']:.1%}")

## 2. Loading Real Results

To use real results, replace `synthetic_results` with the loaded JSON:

In [ ]:
def load_results(path: str) -> dict:
    """Load BeyondBench final_results.json."""
    with open(path) as f:
        return json.load(f)

# Uncomment to use real results:
# results = load_results("./beyondbench_results/final_results.json")

# For this notebook, use synthetic
results = synthetic_results
print("Using synthetic results — replace with load_results() for real data")

## 3. Build Analysis DataFrame

In [ ]:
def extract_task_accuracy(task_name: str, task_data) -> float:
    """Extract accuracy from various result formats."""
    if isinstance(task_data, list) and task_data:
        return sum(m.get("accuracy", 0) for m in task_data) / len(task_data)
    elif isinstance(task_data, dict):
        summary = task_data.get("summary", {})
        return summary.get("avg_accuracy", task_data.get("overall_accuracy",
               task_data.get("accuracy", 0)))
    return 0.0


def get_suite(task_name: str) -> str:
    """Determine which suite a task belongs to."""
    for suite, tasks in SUITE_TASKS.items():
        if task_name in tasks:
            return suite
    return "unknown"


# Build flat DataFrame
rows = []
for task_name, task_data in results["task_results"].items():
    acc = extract_task_accuracy(task_name, task_data)
    rows.append({
        "task": task_name,
        "suite": get_suite(task_name),
        "accuracy": acc,
        "pct": acc * 100,
    })

df = pd.DataFrame(rows).sort_values(["suite", "accuracy"], ascending=[True, False])
print(f"Analysis DataFrame: {df.shape[0]} tasks")
print(df.groupby("suite")["accuracy"].describe().round(3))

## 4. Per-Suite Accuracy Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

suite_colors = {"easy": "#2ecc71", "medium": "#f39c12", "hard": "#e74c3c"}

for ax, suite in zip(axes, ["easy", "medium", "hard"]):
    suite_df = df[df["suite"] == suite].sort_values("accuracy", ascending=True)
    color = suite_colors[suite]

    bars = ax.barh(range(len(suite_df)), suite_df["accuracy"].values,
                   color=color, alpha=0.8, edgecolor="white")
    ax.set_yticks(range(len(suite_df)))
    ax.set_yticklabels(suite_df["task"].values, fontsize=8)
    ax.set_xlim(0, 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.set_title(f"{suite.capitalize()} Suite\n(avg: {suite_df['accuracy'].mean():.1%})",
                 fontsize=12, fontweight="bold")
    ax.axvline(suite_df['accuracy'].mean(), color="darkblue", linestyle="--",
               alpha=0.6, label="Mean")
    ax.grid(axis="x", alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Add value labels
    for bar, val in zip(bars, suite_df["accuracy"].values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.0%}", va="center", fontsize=7)

plt.suptitle(f"BeyondBench Task Accuracy — {results['model_info']['model_name']}",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/tmp/result_analysis_per_suite.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Accuracy Distribution (Violin + Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

suites = ["easy", "medium", "hard"]
data_by_suite = [df[df["suite"] == s]["accuracy"].values for s in suites]
colors = [suite_colors[s] for s in suites]

parts = ax.violinplot(data_by_suite, positions=range(len(suites)), showmeans=True,
                      showextrema=True)

for i, (pc, color) in enumerate(zip(parts['bodies'], colors)):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)

# Overlay individual points
for i, (data, color) in enumerate(zip(data_by_suite, colors)):
    jitter = np.random.uniform(-0.08, 0.08, size=len(data))
    ax.scatter(np.full(len(data), i) + jitter, data,
               color=color, alpha=0.6, s=30, zorder=3)

ax.set_xticks(range(len(suites)))
ax.set_xticklabels([s.capitalize() for s in suites], fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Accuracy Distribution by Suite", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/result_analysis_violin.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top and Bottom Performing Tasks

In [ ]:
N = 8
top_tasks = df.nlargest(N, "accuracy")[["task", "suite", "accuracy"]]
bottom_tasks = df.nsmallest(N, "accuracy")[["task", "suite", "accuracy"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, task_df, title in [
    (ax1, top_tasks, f"Top {N} Tasks"),
    (ax2, bottom_tasks.sort_values("accuracy"), f"Bottom {N} Tasks"),
]:
    bar_colors = [suite_colors.get(s, "gray") for s in task_df["suite"]]
    bars = ax.barh(range(len(task_df)), task_df["accuracy"].values,
                   color=bar_colors, alpha=0.85)
    ax.set_yticks(range(len(task_df)))
    ax.set_yticklabels(
        [f"{t} ({s})" for t, s in zip(task_df["task"], task_df["suite"])],
        fontsize=9
    )
    ax.set_xlim(0, 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for bar, val in zip(bars, task_df["accuracy"].values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.0%}", va="center", fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=suite_colors[s], label=s.capitalize())
                   for s in ["easy", "medium", "hard"]]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10)

plt.suptitle(f"Strongest and Weakest Tasks — {results['model_info']['model_name']}",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.savefig("/tmp/result_analysis_top_bottom.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Statistical Summary

In [ ]:
from scipy import stats

print("=" * 60)
print(f"MODEL: {results['model_info']['model_name']}")
print("=" * 60)

for suite in ["easy", "medium", "hard"]:
    suite_data = df[df["suite"] == suite]["accuracy"]
    print(f"\n{suite.upper()} SUITE ({len(suite_data)} tasks):")
    print(f"  Mean:    {suite_data.mean():.1%}")
    print(f"  Std:     {suite_data.std():.3f}")
    print(f"  Min:     {suite_data.min():.1%}  ({suite_data.idxmin()})")
    print(f"  Max:     {suite_data.max():.1%}  ({suite_data.idxmax()})")
    print(f"  Median:  {suite_data.median():.1%}")
    
    # 95% confidence interval (normal approximation)
    n = len(suite_data)
    se = suite_data.std() / np.sqrt(n)
    ci_low = suite_data.mean() - 1.96 * se
    ci_high = suite_data.mean() + 1.96 * se
    print(f"  95% CI:  [{ci_low:.1%}, {ci_high:.1%}]")

# Overall
all_acc = df["accuracy"]
print(f"\nOVERALL ({len(all_acc)} tasks):")
print(f"  Mean:    {all_acc.mean():.1%}")
print(f"  Std:     {all_acc.std():.3f}")

## 8. Aggregate Over Multiple Runs

When running with multiple folds, aggregate fold results:

In [ ]:
def aggregate_fold_results(task_data) -> dict:
    """
    Aggregate multiple fold results into mean ± std.

    Args:
        task_data: List of fold result dicts, each with an 'accuracy' key

    Returns:
        Dict with mean, std, min, max
    """
    if not isinstance(task_data, list) or not task_data:
        return {"mean": 0, "std": 0, "min": 0, "max": 0}

    accuracies = [m.get("accuracy", 0) for m in task_data]
    return {
        "mean": np.mean(accuracies),
        "std": np.std(accuracies),
        "min": np.min(accuracies),
        "max": np.max(accuracies),
        "n_folds": len(accuracies),
    }


# Simulate multi-fold data (5 folds)
np.random.seed(0)
multi_fold_task = [
    {"fold": i, "accuracy": np.clip(0.85 + np.random.normal(0, 0.03), 0, 1)}
    for i in range(5)
]

agg = aggregate_fold_results(multi_fold_task)
print(f"Multi-fold result:")
print(f"  Mean ± Std: {agg['mean']:.1%} ± {agg['std']:.3f}")
print(f"  Range: [{agg['min']:.1%}, {agg['max']:.1%}]")
print(f"  Folds: {agg['n_folds']}")

## 9. Export Analysis Report

In [ ]:
# Export a CSV summary
summary_path = "/tmp/beyondbench_analysis.csv"
df[["task", "suite", "accuracy"]].sort_values(
    ["suite", "accuracy"], ascending=[True, False]
).to_csv(summary_path, index=False)

print(f"Analysis CSV saved to: {summary_path}")
print()
print("Preview:")
print(df[["task", "suite", "accuracy"]].to_string(index=False)[:2000])

## Summary

You've seen how to:
- Load `final_results.json` from BeyondBench evaluations
- Build a pandas DataFrame for flexible analysis
- Visualize per-suite distributions and top/bottom tasks
- Compute confidence intervals and aggregate multi-fold results
- Export analysis summaries to CSV

**For multi-model comparison**, combine this with the techniques in `03_model_comparison.ipynb`.